In [1]:
import torch
import os

torch.cuda.is_available()

True

In [8]:
def process_raw_article(article):
    """
    Process the raw article data and extract the relevant information.
    :param article: The raw article dict from JSON.
    :return: A dict containing the key fields.
    """
    result = {}
    result["id"] = article["uuid"]
    result["published_at"] = article["first_published_at"]

    content = article["content"]
    result["author"] = content["autor"]
    result["title"] = content["titel"]

    # Extract category
    try:
        category = content["category"]["name"]
    except:
        category = ""
    result["category"] = category

    # Extract section
    try:
        section = content["category"]["content"]["parent"]
    except:
        section = ""
    result["section"] = section

    # Extract the text from the article
    article_text = ""
    for block in content["inhalt"]["content"]:
        if block["type"] == "paragraph":    
            for part in block["content"]:
                if part.get("type") == "text":
                    text = part.get("text", "")
                    article_text += text
                    article_text += "\n"

    result["text"] = article_text
    return result

def create_static_metadata(article, filename):
    """
    Create a static metadata dict for the article.
    :param article: The article dict.
    :param filename: The filename of the article.
    :return: A dict with the metadata.
    """
    article_metadata = {
        "id": article["id"],
        "title": article["title"],
        "author": article["author"],
        "published_at": article["published_at"],
        "words_count": len(article["text"].split(" ")),
        "filename": filename,
        "category": article["category"],
        "section": article["section"],
    }
    return article_metadata

In [2]:
ARTICLES_RAW_DIR = "/export/share/krausef99dm/AIM_delete/data/2025_02_25_wienerzeitung_archiv/content"
ARTICLES_CLEAN_DIR = "/export/share/krausef99dm/AIM_delete/data/2025_02_25_wienerzeitung_archiv/content"

In [27]:
def process_article_file(article_file):
    article_path = os.path.join(ARTICLES_RAW_DIR, article_file)
    # Read the raw article
    with open(article_path, "r", encoding="utf-8") as file:
        article = json.load(file)
    
    # Process the article
    article_clean = process_raw_article(article)
    
    # Write the cleaned article
    clean_path = os.path.join(ARTICLES_CLEAN_DIR, article_file)
    with open(clean_path, "w", encoding="utf-8") as file:
        json.dump(article_clean, file, indent=4, ensure_ascii=False)
    return article_file

In [28]:
# Use ThreadPoolExecutor for concurrent processing
#from concurrent.futures import ThreadPoolExecutor, as_completed
#articles_raw = os.listdir(ARTICLES_RAW_DIR)
#with ThreadPoolExecutor(max_workers=8) as executor:
#    futures = {executor.submit(process_article_file, article_file): article_file for article_file in articles_raw}
    
#    for future in tqdm(as_completed(futures), total=len(futures)):
#        try:
#            file_name = future.result()
#        except Exception as e:
#            print(f"Error processing {futures[future]}: {e}")

100%|██████████| 87754/87754 [01:33<00:00, 941.43it/s] 


In [3]:
import os
import glob
import json
import torch
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from nltk.corpus import stopwords
import string
cache_dir="/path/to/my/huggingface/cache"
# --- Configuration ---
DATA_DIR = "/export/share/krausef99dm/AIM_delete/data/2025_02_25_wienerzeitung_archiv/content"
OUTPUT_DIR = "/export/share/krausef99dm/AIM_delete/data/2025_02_25_wienerzeitung_archiv/EXTRA"   # Folder to store batch output files
BATCH_SIZE = 500               # Adjust batch size according to memory constraints
MAX_LENGTH = 512
CANDIDATE_LABELS = {
    'Finanzkrise': "Finanzkrise",
    'Nachhaltigkeit': "Nachhaltigkeit Umweltschutz Ressourcenschonung Zukunftsfähigkeit Sustainability",
    'Fake News': "Fake News Desinformation Falschnachrichten Irreführung",
    'Künstliche Intelligenz': "Künstliche Intelligenz KI Maschinelles Lernen Deep Learning Automatisierung Artificial Intelligence openAI",
    'Digitalisierung': 'Digitalisierung Digitaler Wandel digitale Transformation Technologisierung',
    'lokaler Journalismus': 'lokaler Journalismus Regionaljournalismus Stadtjournalismus kommunaler Journalismus',
    'COVID': "COVID Coronavirus Pandemie SARS-CoV-2 Corona-Pandemie",
    'Demografie': "Demografie Bevölkerungsentwicklung Altersstruktur demographischer Wandel",
    'Innovation': "Innovation Neuerung Erneuerung technologischer Fortschritt"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# --- Setup device and model ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = 'joeddav/xlm-roberta-large-xnli'
nli_model = AutoModelForSequenceClassification.from_pretrained(model_name, cache_dir="/export/share/krausef99dm/AIM_delete").to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [12]:
# --- Cleaning function ---
stop_words = set(stopwords.words("german"))
def clean_text(text, max_length=5000):
    text = text.lower()
    text = "".join([char if char not in string.punctuation else " " for char in text])
    words = text.split()
    words = [word for word in words if word not in stop_words]
    cleaned = " ".join(words)
    return cleaned[:max_length]

# --- Helper function to process a batch ---
def process_batch(file_list):
    results = {}
    premises = []
    ids = []
    # Load and clean texts
    for file_path in file_list:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            article_id = data.get('id')
            text = data.get('text', '')
            if not text or article_id is None:
                continue
            cleaned = clean_text(text)
            
            #ids.append(article_id)
            ids.append(file_path.split("/")[-1])
            
            premises.append(cleaned)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue

    if not premises:
        return results

    batch_size = len(premises)
    # Create a list of input pairs for all candidate labels per text.
    # The model expects (premise, hypothesis) pair.
    input_pairs = []
    for text in premises:
        for labels in CANDIDATE_LABELS.keys():
            hypothesis = f"Dieser Artikel handelt von {labels}"
            input_pairs.append((text, hypothesis))
    
    # Tokenize in batch
    encodings = tokenizer.batch_encode_plus(
        input_pairs, 
        return_tensors='pt', 
        truncation=True, 
        max_length=MAX_LENGTH,
        padding=True
    )
    encodings = {k: v.to(device) for k, v in encodings.items()}
    
    # Forward pass
    with torch.no_grad():
        logits = nli_model(**encodings)[0]
    
    # We discard the "neutral" (dim 1) and take probability for "entailment" (dim 2)
    entail_contradiction_logits = logits[:, [0, 2]]
    probs = torch.softmax(entail_contradiction_logits, dim=1)
    # Reshape: each text has len(CANDIDATE_LABELS) predictions
    probs = probs[:, 1].reshape(batch_size, len(CANDIDATE_LABELS)).cpu().tolist()
    
    # Map probabilities to each article id and candidate labels
    for i, article_id in enumerate(ids):
        results[article_id] = {label: prob for label, prob in zip(CANDIDATE_LABELS.keys(), probs[i])}
    return results

# --- Main processing loop ---
failed_batches = []    
# Get all JSON files
#file_paths = sorted(glob.glob(os.path.join(DATA_DIR, '*.json')))
file_paths = file_paths_reduced
# vvv FIXME
#import random
#random.seed(42)
#random.shuffle(file_paths)

total_files = len(file_paths)

#total_files = BATCH_SIZE * 2  # FIXME
# ^^^ FIXME 

num_batches = (total_files + BATCH_SIZE - 1) // BATCH_SIZE 

# Process each batch
for batch_idx in tqdm(range(num_batches)):
    output_file = os.path.join(OUTPUT_DIR, f"batch_{batch_idx}.json")
    # Skip batch if already processed
    if os.path.exists(output_file):
        print(f"Skipping batch {batch_idx} (already processed).")
        continue # FIXME

    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, total_files)
    batch_files = file_paths[start:end]
    print(f"Processing batch {batch_idx} (files {start} to {end-1})...")
    
    try:
        batch_results = process_batch(batch_files)
        # Save intermediate results
        with open(output_file, 'w', encoding='utf-8') as f_out:
            json.dump(batch_results, f_out, indent=2)
    except Exception as e:
        print(f"Error in batch {batch_idx}: {e}")
        failed_batches.append(batch_idx)
        # Optionally, log the error or break/continue
        continue
        
#    if batch_idx >= MAX_ITER:
#        break


if failed_batches:
    print("Failed batches:", failed_batches)

  0%|          | 0/1 [00:00<?, ?it/s]

Processing batch 0 (files 0 to 228)...


100%|██████████| 1/1 [00:02<00:00,  2.29s/it]

Error in batch 0: CUDA out of memory. Tried to allocate 14.06 GiB. GPU 0 has a total capacity of 39.39 GiB of which 809.75 MiB is free. Process 3726927 has 414.00 MiB memory in use. Process 3726928 has 414.00 MiB memory in use. Including non-PyTorch memory, this process has 37.76 GiB memory in use. Of the allocated memory 24.96 GiB is allocated by PyTorch, and 12.31 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Failed batches: [0]


In [6]:
failed_files = ['007-und-die-lizenz-zum-verspiessern.json',
 '1-coffee-to-berlin.json',
 '1-euro-wasser-auf-flughafen-die-neueste-errungenschaft-der-eu.json',
 '1-frau-an-der-spitze-der-eliteschule-ena.json',
 '1-mai-arbeitslosigkeit-und-neuwahlen.json',
 '1-mai-es-lebe-die-sozialdemokratie.json',
 '1-mai-schwere-ausschreitungen-in-hamburg.json',
 '1-mai-tag-der-arbeitslosigkeit.json',
 '1-mai-tag-der-erwerbs-arbeit.json',
 '1-mrd-euro-30000-betriebe-die-notwendigkeit-der-verdoppelung.json',
 '1-osterreichische-personalfachmesse.json',
 '1-regierung.json',
 '10-bei-ogh-vs-kicker.json',
 '10-bucher-die-ich-nicht-einmal-geschenkt-haben-will.json',
 '10-euro-eintritt-fur-neos-casting-panne-bei-stronach.json',
 '10-fur-den-grantscherm.json',
 '10-im-match-nachhaltigkeit-gegen-kommerz.json',
 '10-jahre-europlakat-international.json',
 '10-jahre-fair-trade-der-richtige-weg.json',
 '10-jahre-franz-loschnak.json',
 '10-jahre-nach-fukushima.json',
 '10-milliarden-fur-steuerreform.json',
 '10-millionen-neue-jobs-in-5-jahren.json',
 '10-prozent-mehr-todesfalle-durch-corona.json',
 '10-tipps-fur-sicheres-weihnachts-online-shopping.json',
 '10-years-a-refugee.json',
 '100-erfindungen-fur-die-welt.json',
 '100-geburtstag-von-politlegende-franz-olah.json',
 '100-grune-auf-hausbesuch.json',
 '100-jahre-fluch.json',
 '100-jahre-insulin-ein-meilenstein.json',
 '100-jahre-klimabundnis-journalismus-und-wissenschaft.json',
 '100-jahre-nordirland-schwere-last-im-brexit-poker.json',
 '100-jahre-pr-320-jahre-wiener-zeitung.json',
 '100-jahre-richtervereinigung.json',
 '100-jahre-ungarn-traum-drama-trauma.json',
 '100-jahre-verfassung-und-der-koran.json',
 '100-jahre-verfassung.json',
 '100-jahrige-fit-wie-50er.json',
 '100-milliarden-euro-fur-osterreichs-banken.json',
 '100-milliarden-euro-gegen-bankenkrise.json',
 '100-millionen-euro-an-mehrkosten-durch-antrage-im-parlament.json',
 '100-millionen-euro-fur-die-pflege.json',
 '100-millionen-euro-fur-osterreich-liegen-brach.json',
 '100-millionen-fur-klimaschutz-kandidaten.json',
 '100-millionen-sind-kein-klacks.json',
 '100-plus-2-ist-mirek-topolaneks-gluckszahl.json',
 '100-sekunden-vor-12.json',
 '100-tage-ferien-im-ersten-arbeitsjahr.json',
 '100-tage-frist-fur-die-rettung-der-weltfinanzen.json',
 '100-tage-gehen-sich-aus.json',
 '100-tage-kung.json',
 '100-tage-regierung-die-bilanz-kann-sich-sehen-lassen.json',
 '100-tage-streit-deutsche-sind-mit-wunschkoalition-nicht-zufrieden.json',
 '1000-euro-fur-die-wurde.json',
 '1000-euro-mindestlohn-nun-fixiert.json',
 '1000-nachte-weg-von-zuhause.json',
 '1000-peitschenhiebe-fur-eine-meinung.json',
 '1000-peitschenhiebe-und-zehn-jahre.json',
 '1000-polizisten-in-4-jahren.json',
 '1000-studenten-fur-die-praterstrasse.json',
 '10000-anwalte-wachen-uber-wahlen-in-den-usa.json',
 '10000-leserbriefe.json',
 '10000-nato-soldaten-konnen-nicht-das-ganze-land-sichern.json',
 '10000-syrische-babys-geboren-in-einem-anderen-land.json',
 '100000-platze-mehr-in-zehn-jahren.json',
 '1001-nacht-in-ithaka.json',
 '1001-nacht.json',
 '1001-wege-der-rechtsfindung.json',
 '101-beschuldigte-bei-verfahren-gegen-muslimbruder.json',
 '101-dalmatiner-lachen-dreckig.json',
 '104-mal-gefalscht.json',
 '108781-asylwerber-im-vorjahr-der-grossteil-zog-weiter.json',
 '11-bewerber-um-unesco-fuhrung.json',
 '11-mrd-s-aus-steuertopf.json',
 '11-september-2001-der-tag-an-dem-die-welt-den-atem-anhielt.json',
 '11-september-2001-stunden-des-horrors.json',
 '1100-neue-autos-fur-bundespolizei.json',
 '11000-freiwillige-fur-attentate.json',
 '111-tage-fur-ein-neues-system.json',
 '1110-auslander-wollen-nach-anschlagen-aufenthaltsrecht.json',
 '1137-gegen-den-videobeweis.json',
 '115-sekunden-spezialausbildung.json',
 '1161-lehrerposten-fur-sonderforderung.json',
 '117-kardinale-einer-wird-papst.json',
 '117-kardinale-sind-bereit-fur-das-nachste-konklave.json',
 '119-milliarden-fur-die-infrastruktur.json',
 '12-februar-1934-kuschelkurs-schlagt-burgerkrieg.json',
 '12-fur-die-kids-60-fur-die-fischrsquo.json',
 '12-jahriger-prophezeite-unruhen.json',
 '12-millionen-boni-fur-pleitiers.json',
 '12-mrd-euro-mehr-fur-forschung.json',
 '12-stunden-tag-hat-dort-nichts-verloren.json',
 '12-stunden-tag-neue-sanktionen.json',
 '12-stunden-tag-wird-kommen.json',
 '120-reformer-treten-zuruck.json',
 '1200-km-zeit-zum-abschalten.json',
 '1200-neue-lehrplatze.json',
 '122-milliarden-euro-fur-die-strasse.json',
 '122-postenschliessungen-ersparen-null-cent.json',
 '1226-planstellen-mehr-im-bundesdienst-die-halfte-davon-bei-der-polizei.json',
 '1244-stimmen-fur-einen-staatschef.json',
 '125-jahre-parlamentsgebaude.json',
 '125-jahre-und-ein-bisschen-leiser.json',
 '125-millionen-fur-halbe-kommunalkredit-austria.json',
 '127-community-medien-zu-migration-und-diversitat.json',
 '1284-neue-planeten.json',
 '13-facetten-eines-stellvertreterkrieges.json',
 '13-jahre-fur-die-ermordung-einer-siebenjahrigen.json',
 '13-jahre-fur-ein-neues-medikament.json',
 '13-jahre-nervenkrieg-im-atomstreit.json',
 '13-lowen-gingen-nach-grossbritannien.json',
 '13-milliarden-fur-osterreich.json',
 '13-milliarden-gegen-den-energiekosten-horror.json',
 '13-millionen-agypter-wahlten-mursi-22-millionen-sturzten-ihn.json',
 '13-millionen-krisenhilfe-fur-armere.json',
 '13-prozent-in-den-sattel-bekommen.json',
 '13-tote-bei-golan-ersturmung.json',
 '13-turkische-soldaten-tot-konflikt-mit-pkk-eskaliert.json',
 '130-jahre-gebildete-wiener.json',
 '130-spitaler.json',
 '133-neuankommlinge.json',
 '1333-jahre-unversohnlicher-hass-zwischen-sunniten-und-schiiten.json',
 '134-nobelpreistrager-fur-liu-xiaobo.json',
 '134-tankstellen-fur-nur-4000-erdgas-fahrzeuge.json',
 '1357-delegierten-stimmen-sind-zu-holen.json',
 '137500-euro-pro-meter-autobahn.json',
 '138-euro-mindestlohn.json',
 '14-gemeinden-erheben-schwere-vorwurfe-gegen-raiffeisen.json',
 '14-mal-siebenbrunn.json',
 '14-mal-so-viel-fur-besserverdiener.json',
 '14-von-19-kandidaten-durfen-nicht-antreten.json',
 '140-jahre-zu-leben-ist-gar-nicht-leicht.json',
 '140-millionen-vom-bund-sind-zu-wenig.json',
 '140000-wollen-die-post-retten.json',
 '142-milliarden-euro-brexit-loch.json',
 '145-betroffene-von-sexueller-belastigung-in-ministerien.json',
 '145-mill-euro-frisches-kapital-fur-rzb.json',
 '15-erden-braucht-der-mensch.json',
 '15-jahre-danach-von-jubel-keine-spur.json',
 '15-jahre-eu-erweiterung-eine-bilanz.json',
 '15-jahre-friedenseinsatz-im-kosovo.json',
 '15-langzeitarbeitslose-sind-corona-helfer-in-karnten.json',
 '15-meter-abstand-zu-anderen-sind-das-minimum.json',
 '15-milliarden-dollar-fur-eine-zukunft.json',
 '15-milliarden-fur-schnelleres-internet-aber-nur-schrittweise.json',
 '15-milliarden-schwarz-gebunkert.json',
 '15-millionen-afghanen-demnachst-auf-der-flucht.json',
 '15-millionen-an-nur-ein-institut.json',
 '15-prozent-infizierte-in-deutschlands-ischgl.json',
 '15-prozent-weniger-verkehrstote.json',
 '15-sekunden-zwischen-hypes-und-fuck-ups.json',
 '150-euro-pro-stunde-furs-impfen-stolze-summe.json',
 '150-facher-morder-und-kronzeuge-gegen-die-mafia.json',
 '150-jahre-cartellverband.json',
 '150-jahre-postkarte-analoge-grusse-bleiben-aktuell.json',
 '150-jahre-staatsoper-buhne-der-stimmgiganten.json',
 '150-tage-danach-koalition-will-auch-im-herbst-ein-hohes-reformtempo-fahren.json',
 '1500-euro-fur-die-alte-klapperkiste.json',
 '1500-kilometer-zum-deutschkurs.json',
 '15000-jobangebote-fur-berufsein-und-umsteiger.json',
 '15000-milliarden-dollar-schulden-und-keine-losung.json',
 '15000-ordinationen-im-test.json',
 '150000-euro-strafe-fur-marktmanipulation.json',
 '152-tage-politische-agonie.json',
 '15500-stunden-langer-test-fur-sparlampen.json',
 '15600-sind-eher-eine-lotteriezahl.json',
 '159-kritische-strassennamen.json',
 '16-eib-milliarden-fur-osterreich.json',
 '16-millionen-euro-fur-corona-schutz-fur-polizisten.json',
 '16-millionen-italienische-wahler-wechselten.json',
 '160-gemeinden-wollen-fusionieren.json',
 '1600-juden-gesteinigt-und-verbrannt.json',
 '1600-juden-von-polen-am-10-juli-1941-in-jedwabne-massakriert.json',
 '16000-nokia-mitarbeiter-bangen.json',
 '16000-osterreichern-ohne-pflichtschule-droht-weniger-sozialhilfe.json',
 '163000-einwanderer-494-jobs.json',
 '17-ex-kommissare-verdienen-in-der-pension-doppelt.json',
 '17-experten-sollen-italien-retten.json',
 '17-februar-2012.json',
 '17-jahr-mandatar-politik-fur-schuler.json',
 '17-jahrige-aufs-bike-15-jahrige-aufs-moped.json',
 '17-millionen-tweets-pro-tag.json',
 '17-sitze-sichern-scheuch.json',
 '17-tote-bei-sturm-auf-gefangnis.json',
 '17-unterirdische-pyramiden-entdeckt.json',
 '17.json',
 '170-fragen-zum-brexit.json',
 '170-tage-auf-der-verregneten-insel.json',
 '1700-euro-lohn-vor-kurzerer-arbeitszeit.json',
 '1700-euro-steuerfrei-was-der-spo-plan-bringt.json',
 '1700-lehrlinge-ohne-ausbildung.json',
 '1700-polizisten-fur-eine-hausraumung.json',
 '17000-kleinanleger-klagen-an.json',
 '17000-seiten-uber-ein-verbrechen.json',
 '1744-milliarden-euro.json',
 '176-millionen-euro-aus-klagenfurter-stadtkasse-weg.json',
 '18-eu-mandatare-inexistent.json',
 '18-jahre-in-haft-mein-sohn-kommt-nach-hause.json',
 '18-milliarden-entlastung-2023.json',
 '60-jahre-romische-vertrage.json',
 'bilder-einer-flucht-ins-ungewisse.json',
 'bio-boom-in-osterreich-halt-an.json',
 'contra-sedierung-des-landes-in-ubergrosser-koalition-aufbrechen.json',
 'das-regierungsprogramm-als-begriffswolke.json',
 'dd.json',
 'debatte-mit-sechs-spitzenkandidaten-im-europaparlament.json',
 'der-herzigste-fisch-der-welt.json',
 'die-lage-fur-hilfesuchende-menschen-spitzt-sich-zu.json',
 'die-medienkompetenz-der-bundesregierung.json',
 'die-welt-aus-sicht-der-iss.json',
 'ein-jein-gibt-es-nicht.json',
 'einblick-in-die-brustkrebsforschung.json',
 'fluchthilfe-aus-osterreich.json',
 'franz-voves-ein-politikerleben-in-bildern.json',
 'gedenken-an-die-opfer-von-auschwitz.json',
 'geloscht.json',
 'haupls-grosser-abschiedsauftritt.json',
 'im-schutt-gibt-es-immer-noch-sprengkorper.json',
 'impressionen-von-der-automesse-iaa.json',
 'international.json',
 'kurz-besucht-berlin-deutsche-medien-kommentieren-mit-biss.json',
 'new-york-am-wahltag.json',
 'nicht-freischalten.json',
 'osterreichs-schulen-als-interaktive-grafik.json',
 'sprengt-die-blase.json',
 'toiletten-in-wien-anstalten-fur-alle-falle.json',
 'urban-gardening-wiens-guerilla-grunoasen.json',
 'von-fluchtlingsstromen-und-un-menschlichkeit.json']

In [8]:
file_paths = sorted(glob.glob(os.path.join(DATA_DIR, '*.json')))
file_paths_reduced = [i for i in file_paths if i.split("/")[-1] in failed_files]

In [9]:
len(file_paths_reduced)

229